# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*I'm choosing Lane 2: Refresh / Content Opportunity Scoring. The question is:
which pages should be reviewed first for refresh, based on evidence and
limited review capacity? I picked this lane because the starter pipeline
already runs this exact workflow end-to-end, and Week 1 showed a real,
learnable signal in it -- a random forest model clearly outperformed a
transparent hand-written baseline rule on this task. That gap is a strong
starting signal that this lane is worth building on for the next 7 weeks.*

In [14]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/Flyrank-ML-Internship/Flyrank-ML-Internship/Flyrank-ML-Internship
Starter data found. You're ready.


## 2. The question: decision, action, cost of a wrong call

Question: Which pages should a content team prioritize for refresh,
expansion, or protection review?

- Unit of analysis: one page (content_id), evaluated using its prior
  90-day window of observed signals.
- Decision improved: which of many pages a content/SEO reviewer spends
  their limited hours on first, this cycle.
- Who acts: a content strategist or SEO reviewer with a fixed review
  capacity (e.g. can only review the top 50 pages this month).
- Cost of a wrong call:
  - False positive: hours spent refreshing a page that wasn't actually
    declining -- wasted reviewer time, no real content team resource is
    unlimited.
  - False negative: a truly declining page gets skipped and keeps losing
    visibility for weeks before anyone notices.
  Because capacity is limited, Precision@K (e.g. Precision@50) is the
  right metric -- it matches how the ranked list will actually be used:
  a reviewer works through the top of the list, not the whole inventory.
This is decision-support, not a guarantee -- a refresh being recommended
does not mean it is guaranteed to cause recovery; that would require a
causal experiment this data alone can't provide.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/Flyrank-ML-Internship/Flyrank-ML-Internship/Flyrank-ML-Internship/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/Flyrank-ML-Internship/Flyrank-ML-Internship/Flyrank-ML-Internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/Flyrank-ML-Internship/Flyrank-ML-Internship/Flyrank-ML-Internship/data/processed/model_predictions.csv
Wrote model results: /content/Flyrank-ML-Internship/Flyrank-ML-Internship/Flyrank-ML-Internship/outputs/model_results.json

▶ Step 4/5 — Evalua

## 3. Quick look at the data (2-3 real numbers)

Three real numbers back this lane:
1. Baseline rule Precision@50 = 0.240 (~12 of top 50 correct) vs random
   forest Precision@50 = 0.740 (~37 of top 50 correct) -- roughly a 3x
   lift, confirming this is a learnable problem, not random noise.
2. Random forest ROC AUC = 0.750 -- a real discriminative signal, well
   above the 0.5 baseline of random guessing.
3. This result comes from a 30,000-row anonymized starter slice with
   client-holdout validation (whole clients kept out of training) -- so
   the model was tested on clients it had never seen, which is a
   meaningful early validation signal.

*Loading the starter CSV and showing 2-3 real numbers that make my lane look worth the next 7 weeks.*

In [16]:
import pandas as pd, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
res = json.load(open("outputs/model_results.json"))

base_p50 = res["baseline"]["baseline_precision_at_50"]
rf_p50   = res["models"]["random_forest"]["precision_at_50"]
rf_auc   = res["models"]["random_forest"]["roc_auc"]

print(f"Rows: {df.shape[0]}")
print(f"Baseline rule Precision@50: {base_p50:.3f}  (~{round(base_p50*50)} of top 50 right)")
print(f"Random forest Precision@50: {rf_p50:.3f}  (~{round(rf_p50*50)} of top 50 right)")
print(f"Random forest ROC AUC: {rf_auc:.3f}")
print(f"Lift over baseline: {rf_p50/base_p50:.1f}x")

Rows: 30000
Baseline rule Precision@50: 0.240  (~12 of top 50 right)
Random forest Precision@50: 0.740  (~37 of top 50 right)
Random forest ROC AUC: 0.750
Lift over baseline: 3.1x


## 4. Careful words: what I can and can't claim

What I can claim: observed, measured patterns in this anonymized sample --
e.g. "these signals were associated with pages later bucketed as
declining in this dataset." My final output will be framed as directional
and decision-support: a ranked review queue to help a human reviewer
prioritize limited time, with reason codes they can inspect and
disagree with.

What I will never claim:
- That refreshing a page CAUSES recovery -- that needs a real experiment.
- That I've reverse-engineered or predicted Google's ranking algorithm.
- That a correlation between a signal and decline proves that signal
  is the reason a page declined.
- I will also note the current starter label (trend_direction == "down")
  is a proxy from the current window, not a true future outcome -- a
  weakness I plan to improve toward a real prior-window -> future-window
  label as the lane guide recommends.)

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.